# Create Volkswagen Foundation Awards (VolkswagenStiftung)

Creates Volkswagen Foundation awards from the foundation's own public project
database, the **Projektdatenbank** at https://projektdatenbank.volkswagenstiftung.de/
("contains most research projects funded since 2004"). It is a Blazor Server
app whose pages are server-prerendered; `sitemap.xml` enumerates every project
page. Each project page carries Projektnummer, summary, keywords, status,
start/end date, Foerdersumme (EUR), profile area, funding initiative, call and
the project lead(s) with institution and optional ORCID. The project title
comes from the leads' person pages (project cards). **4,320 projects, start years 2004-2027, 99.9% title, 100% amount (EUR 1.84bn total), 100% lead PI + institution, 25% lead ORCID** (local run 2026-09-30).

No export/CSV/API exists (runbook ladder item 0 checked 2026-09-30). The
tracker's "30k+ projects" was wrong: the database lists ~4,300 projects.

**Prerequisites:**
- Run `scripts/local/volkswagen_foundation_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/volkswagen_foundation/volkswagen_foundation_projects.parquet`
  (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the database's `Projektnummer`, shipped exactly
as printed (7 characters with leading zeros, e.g. `0203030`, or `N000004` for
some 2007-2011 fellowships). Citation-side sample (crossref_work.grants /
crossref_work_funders for F4320320882, 2026-09-30): most citing works quote the
foundation's older **Aktenzeichen** (e.g. `Az. 94 758`, `I/84 794`, `A131968`,
`9B937`, ZN-numbers), which the Projektdatenbank does not publish, so those
~3,300 stubs cannot collapse onto these rows. The ~29 citations that quote the
7-digit project number (e.g. `0065316`, `0200187`) do match. Keeping the
funder's own public reference is the §2.1.1 fallback.

**Description:** the prerender truncates long summaries at ~750 characters
("..."; the full text needs an interactive click). Shipped as published;
`summary_truncated` in the raw table flags them.

**Amounts:** `Foerdersumme` in EUR (German number format parsed in the script); the 32 projects listed with 0 EUR ship amount NULL.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320320882`
- display_name: Volkswagen Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 03bsmfz84, doi 10.13039/501100001663)
- Not to be confused with Volkswagen AG (F4320329929) or Volkswagen of America.

**Priority:** `488` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.volkswagen_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/volkswagen_foundation/volkswagen_foundation_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.volkswagen_foundation_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.volkswagen_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.volkswagen_foundation_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320882 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320320882;

## Step 2: Create Volkswagen Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.volkswagen_foundation_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320882  -- Volkswagen Foundation
),

people AS (
    -- Project leads in page order -> investigator structs.
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        project_number,
        TRANSFORM(COALESCE(people, ARRAY()), p -> named_struct(
            'given_name', NULLIF(TRIM(p.given_name), ''),
            'family_name', NULLIF(TRIM(p.family_name), ''),
            'orcid', CASE WHEN p.orcid RLIKE '^[0-9]{4}-' THEN CONCAT('https://orcid.org/', p.orcid) END,
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(p.institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        )) AS investigators
    FROM openalex.awards.volkswagen_foundation_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.project_number))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.summary), '') as description,
    f.funder_id,
    TRIM(g.project_number) as funder_award_id,
    NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) as amount,
    CASE WHEN NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    -- Foerderinitiative (e.g. "Momentum - Foerderung fuer Erstberufene", "Opus Magnum").
    COALESCE(NULLIF(TRIM(g.funding_initiative), ''), NULLIF(TRIM(g.call), '')) as funder_scheme,
    'volkswagen_foundation_projektdatenbank' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    -- try_element_at: safe on empty arrays (§2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.project_number))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.volkswagen_foundation_raw g
JOIN people p ON p.project_number = g.project_number
CROSS JOIN src_funder f
WHERE g.project_number IS NOT NULL AND TRIM(g.project_number) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): funder_award_id is the 7-character Projektnummer, unique.
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT id) AS distinct_ids,
    SUM(CASE WHEN funder_award_id RLIKE '^(0[0-9]{6}|N[0-9]{6})$' THEN 0 ELSE 1 END) AS bad_shape
FROM openalex.awards.volkswagen_foundation_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'volkswagen_foundation_projektdatenbank' AND priority = 488;

-- Insert into openalex_awards_raw with priority.
-- Priority 488: direct-from-funder ingest of the VolkswagenStiftung Projektdatenbank.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    488 as priority
FROM openalex.awards.volkswagen_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.orcid,
       lead_investigator.affiliation.name
FROM openalex.awards.volkswagen_foundation_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.volkswagen_foundation_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 50;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.volkswagen_foundation_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.volkswagen_foundation_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.volkswagen_foundation_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.orcid) as has_pi_orcid,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.volkswagen_foundation_awards;

In [ ]:
%sql
-- Overlap with existing citation-side stubs for this funder (expect small: most cite the old Aktenzeichen).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as matched_by_this_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.volkswagen_foundation_awards t ON t.id = c.id
WHERE c.funder_id = 4320320882 AND c.priority < 3
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'volkswagen_foundation_projektdatenbank'
GROUP BY provenance, priority;